# Related Work and Sequential Model Selection

**Dataset:** [Swahili News Classification (Zindi)](https://zindi.world/competitions/swahili-news-classification)

This notebook reviews related research and uses it, together with our data analysis, to choose five approaches: two simple baselines and three neural sequence models.



## 1. Related Work

### 1.1 The problem
The dataset contains Swahili articles from Tanzanian news websites, labelled with five categories (David, 2020). Swahili has over 100 million speakers but few labelled datasets. Classifying news automatically helps news sites sort and recommend articles, and is a building block for other Swahili language tools.

### 1.2 News classification in African languages
**MasakhaNEWS** (Adelani et al., 2023) compared simple models and pretrained transformers on news classification in 16 African languages, including Swahili. Two results matter for us:
- Transformers pretrained on African languages gave the best results.
- Simple keyword-based models were often close behind, because topic words are strong clues.

**AfriBERTa** (Ogueji et al., 2021) is one of these African-language transformers. It was pretrained on 11 African languages including Swahili, and matched much larger multilingual models. It is small enough to fine-tune on a free Colab GPU.

**SwahBERT** (Martin et al., 2022) was tested on a larger, six-category version of this news dataset (same source, same author). A model pretrained only on Swahili beat multilingual BERT, which confirms that a model that already knows Swahili has an advantage on this task.

### 1.3 Ways to model word order
| Model | Key paper | How it uses word order |
|---|---|---|
| TF-IDF + simple classifier | Wang & Manning (2012) | Ignores it: only counts words |
| CNN | Kim (2014) | Spots short phrases of a few words, anywhere in the text |
| LSTM | Hochreiter & Schmidhuber (1997) | Reads word by word and keeps a memory of what came before |
| Transformer | Vaswani et al. (2017) | Every word looks at every other word at once |

### 1.4 Subword tokens
Swahili builds words by adding pieces to a root: *a-li-sema* ("he/she said"), *wa-na-sema* ("they are saying"). **Subword tokenisation** (Sennrich et al., 2016) splits words into these reusable pieces, so a rare word is still made of known parts. Because 53.5% of words appear only once, all three neural models use AfriBERTa's subword tokeniser.

## 2. The Five Approaches

### Approach 1 (baseline): TF-IDF + Logistic Regression
- **What:** scores how important each word is in an article (TF-IDF), then learns which words point to which category.
- **Why:** each category has clear keywords, and MasakhaNEWS found this kind of model to be a strong baseline. It is fast, and its word weights show what it learned.
- **Limitation:** ignores word order and context. *Simba* the football club and *simba* the animal look the same.

### Approach 2 (baseline): TF-IDF + Naive Bayes
- **What:** learns how often each word appears in each category, then picks the most likely category for a new article.
- **Why:** the classic text classification baseline (Wang & Manning, 2012). It learns differently from logistic regression, so comparing the two shows how much the choice of simple model matters.
- **Limitation:** assumes words are independent of each other, and tends to favour the large classes.

### Approach 3 (neural): 1D CNN
- **What:** slides a small window (e.g. 5 tokens) along the article to detect useful phrases, then keeps the strongest signal for each (Kim, 2014).
- **Why:** topics are often signalled by short phrases such as *Ligi Kuu Tanzania* or *Benki Kuu*, which a CNN finds wherever they appear. It is also fast on long articles.
- **Limitation:** only sees a few tokens at a time, so it misses longer context.

### Approach 4 (neural): Bidirectional LSTM
- **What:** reads the article token by token, forwards and backwards, keeping a memory of what it has read (Hochreiter & Schmidhuber, 1997).
- **Why:** articles are long (median 271 words), and the topic can depend on context spread across several sentences, which the CNN's short window cannot see. The LSTM carries a memory through the whole article, so comparing it with the CNN tests whether longer context helps more than short phrases.
- **Limitation:** its memory still fades over hundreds of tokens, so on long articles it may not beat the CNN. It is also slow, because it reads one token at a time, and learns only from our ~5k articles.

### Approach 5 (neural): Fine-tuned AfriBERTa
- **What:** a transformer already pretrained on African languages (Ogueji et al., 2021). We add a classification layer and train it further on our articles.
- **Why:** research shows African-language pretrained models do best (MasakhaNEWS). Since it already knows Swahili, it should help most with the small classes (only 54 and 17 articles) and rare words.
- **Limitation:** reads only the first 512 tokens, so the end of long articles is lost, is the slowest to train, and is the hardest to explain.

### Summary

| | 1. TF-IDF + LR | 2. TF-IDF + NB | 3. CNN | 4. BiLSTM | 5. AfriBERTa |
|---|---|---|---|---|---|
| Type | Baseline | Baseline | Neural | Neural | Neural (pretrained) |
| Word order used | None | None | Short phrases | Whole sequence | Whole sequence |
| Input | TF-IDF word scores | TF-IDF word scores | Subword tokens | Subword tokens | Subword tokens |
| Text used | Whole article | Whole article | First 512 tokens | First 512 tokens | First 512 tokens |
| Speed | Very fast | Very fast | Fast | Slow | Slowest |



## References

- Adelani, D. I., et al. (2023). MasakhaNEWS: News topic classification for African languages. *IJCNLP-AACL 2023*. https://aclanthology.org/2023.ijcnlp-main.10/
- David, D. (2020). *Swahili: News Classification Dataset* [Data set]. Zenodo. https://doi.org/10.5281/zenodo.4300294
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
- Kim, Y. (2014). Convolutional neural networks for sentence classification. *EMNLP 2014*. https://aclanthology.org/D14-1181/
- Martin, G. L., Mswahili, M. E., Jeong, Y.-S., & Woo, J. (2022). SwahBERT: Language model of Swahili. *NAACL 2022*. https://aclanthology.org/2022.naacl-main.23/
- Ogueji, K., Zhu, Y., & Lin, J. (2021). Small data? No problem! Exploring the viability of pretrained multilingual language models for low-resourced languages. *MRL Workshop, EMNLP 2021*. https://aclanthology.org/2021.mrl-1.11/
- Sennrich, R., Haddow, B., & Birch, A. (2016). Neural machine translation of rare words with subword units. *ACL 2016*. https://aclanthology.org/P16-1162/
- Vaswani, A., et al. (2017). Attention is all you need. *NeurIPS 2017*. https://arxiv.org/abs/1706.03762
- Wang, S., & Manning, C. D. (2012). Baselines and bigrams: Simple, good sentiment and topic classification. *ACL 2012*. https://aclanthology.org/P12-2018/
- Zindi. (2020). *Swahili News Classification Challenge*. https://zindi.world/competitions/swahili-news-classification